In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

In [ ]:
print("Loading dataset...")
dataset = load_dataset("SetFit/ag_news", split="train[:1000]")

# Extract the text column into a Python list
documents = dataset["text"]

print(f"Loaded {len(documents)} documents.")
print(f"Sample: {documents[0][:100]}...")

Loading dataset...
Loaded 1000 documents.
Sample: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...


In [ ]:
# embeddings
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert text documents into numerical vector embeddings
print("Encoding documents (this may take a few seconds)...")
document_embeddings = model.encode(documents, show_progress_bar=True)

print(f"Created {document_embeddings.shape[0]} embeddings.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding documents (this may take a few seconds)...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Created 1000 embeddings.


In [ ]:
search_engine = NearestNeighbors(n_neighbors=5, metric="cosine")

search_engine.fit(document_embeddings)
print("Search engine is ready!")

Search engine is ready!


In [25]:
def semantic_search(query, top_k=3):
    # Embed the incoming search query
    query_embedding = model.encode([query])

    # Retrieve the closest matches
    distances, indices = search_engine.kneighbors(query_embedding, n_neighbors=top_k)

    print(f"\n🔍 Query: '{query}'")
    print("-" * 50)

    for i in range(top_k):
        doc_idx = indices[0][i]
        # Convert cosine distance to similarity (1 - distance)
        similarity = 1 - distances[0][i]

        print(f"Result {i+1} (Similarity: {similarity:.4f})")
        print(f"Text: {documents[int(doc_idx)][:150]}...\n")

In [ ]:
#result
semantic_search("Wall street and stock market trends")


🔍 Query: 'Wall street and stock market trends'
--------------------------------------------------
Result 1 (Similarity: 0.6258)
Text: Stocks Higher Despite Soaring Oil Prices NEW YORK - Wall Street shifted higher Monday as bargain hunters shrugged off skyrocketing oil prices and boug...

Result 2 (Similarity: 0.5586)
Text: Stocks Sharply Higher on Dip in Oil Prices NEW YORK - A drop in oil prices and upbeat outlooks from Wal-Mart and Lowe's prompted new bargain-hunting o...

Result 3 (Similarity: 0.5459)
Text: Strategies for a Sideways Market (Reuters) Reuters - The bulls and the bears are in this\together, scratching their heads and wondering what's going t...


🔍 Query: 'Space exploration and rocket launches'
--------------------------------------------------
Result 1 (Similarity: 0.5803)
Text: Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the exploration of the Moon and other planets in our solar s...

Result 2 (Similarity: 0.5008)
Text: 

In [ ]:
semantic_search("Space exploration and rocket launches")


🔍 Query: 'Space exploration and rocket launches'
--------------------------------------------------
Result 1 (Similarity: 0.5803)
Text: Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the exploration of the Moon and other planets in our solar s...

Result 2 (Similarity: 0.5008)
Text: Canadian Team Joins Rocket Launch Contest (AP) AP - The  #36;10 million competition to send a private manned rocket into space started looking more li...

Result 3 (Similarity: 0.4724)
Text: The Next Great Space Race: SpaceShipOne and Wild Fire to Go For the Gold (SPACE.com) SPACE.com - A piloted rocket ship race to claim a  #36;10 million...

